In [1]:
import os
import sys
# Force PySpark to use the virtual environment's Python interpreter
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from spark_h3_turbo import *


In [2]:
print("PySpark UDF tests environment configured.")


PySpark UDF tests environment configured.


In [3]:
print("Starting Spark Session...")
spark = SparkSession.builder \
        .appName("H3TurboTest") \
        .master("local[2]") \
        .getOrCreate()
data = [
    (40.6892, -74.0445),  # Statue of Liberty
    (37.7749, -122.4194)  # SF
]
df = spark.createDataFrame(data, ["lat", "lon"])

# 1. Lat/Lon to Cell
print("Testing latlng_to_cell_udf...")
df = df.withColumn("h3", latlng_to_cell_udf(9)(col("lat"), col("lon")))
df.show()

# 2. Cell to Parent
print("Testing cell_to_parent_udf...")
df = df.withColumn("parent", cell_to_parent_udf(5)(col("h3")))
df.show()

# 3. Grid Disk
print("Testing grid_disk_udf...")
df = df.withColumn("kring", grid_disk_udf(2)(col("h3")))
df.show()

# 4. Spatial Join (Broadcast)
print("Testing spatial_join_udf...")
# Get the H3 values to use as zones
rows = df.select("h3").collect()
zones_list = [r[0] for r in rows]
df = df.withColumn("in_zone", spatial_join_udf(zones_list, 9)(col("h3")))
df.show()

# 5. Batch transform
print("Testing batch_transform_udf...")
df = df.withColumn("scrambled", batch_transform_udf(5)(col("h3")))
df.show()

print("All tests passed!")

Starting Spark Session...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/27 06:44:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Testing latlng_to_cell_udf...


+-------+---------+------------------+
|    lat|      lon|                h3|
+-------+---------+------------------+
|40.6892| -74.0445|617733151067471871|
|37.7749|-122.4194|617700169957507071|
+-------+---------+------------------+

Testing cell_to_parent_udf...


[ACPP] Runtime: Backend CUDA (platform 0) is active.
[ACPP] Device: NVIDIA GeForce RTX 4070 SUPER


+-------+---------+------------------+------------------+
|    lat|      lon|                h3|            parent|
+-------+---------+------------------+------------------+
|40.6892| -74.0445|617733151067471871|599718752904282111|
|37.7749|-122.4194|617700169957507071|599685771850416127|
+-------+---------+------------------+------------------+

Testing grid_disk_udf...
+-------+---------+------------------+------------------+--------------------+
|    lat|      lon|                h3|            parent|               kring|
+-------+---------+------------------+------------------+--------------------+
|40.6892| -74.0445|617733151067471871|599718752904282111|[6177331510674718...|
|37.7749|-122.4194|617700169957507071|599685771850416127|[6177001699575070...|
+-------+---------+------------------+------------------+--------------------+

Testing spatial_join_udf...


[ACPP] Runtime: Backend CUDA (platform 0) is active.
[ACPP] Device: NVIDIA GeForce RTX 4070 SUPER


+-------+---------+------------------+------------------+--------------------+-------+
|    lat|      lon|                h3|            parent|               kring|in_zone|
+-------+---------+------------------+------------------+--------------------+-------+
|40.6892| -74.0445|617733151067471871|599718752904282111|[6177331510674718...|      1|
|37.7749|-122.4194|617700169957507071|599685771850416127|[6177001699575070...|      1|
+-------+---------+------------------+------------------+--------------------+-------+

Testing batch_transform_udf...
+-------+---------+------------------+------------------+--------------------+-------+------------------+
|    lat|      lon|                h3|            parent|               kring|in_zone|         scrambled|
+-------+---------+------------------+------------------+--------------------+-------+------------------+
|40.6892| -74.0445|617733151067471871|599718752904282111|[6177331510674718...|      1|599718752904282111|
|37.7749|-122.4194|617